# Lee, Matthews, Lemieux et al. (2025) on dandiset 001195

Paper: Lee, Matthews, Lemieux et al., "Separable Dorsal Raphe Dopamine Projections mediate the Facets
of Loneliness-like state", eLife (2025), doi:10.7554/eLife.105955. The claims file
(`claims/001195/10.7554-elife.105955.yaml`) was written from the eLife reviewed preprint, version 2
(8 July 2025), with its Methods, legends and figure supplements.

Dataset: [DANDI:001195](https://dandiarchive.org/dandiset/001195/0.250408.1733), version
0.250408.1733 (doi:10.48324/dandi.001195/0.250408.1733). It is the only published version, and the
draft has the same 230 assets and size, so it is pinned. The paper cites no version.

The dandiset holds 148 ex vivo patch-clamp cells (one NWB file each) and 41 in vivo imaging
sessions of 14 mice (CNMF-E traces, footprints, the raw one-photon movie and the behaviour video as
an external mp4). It holds no pose estimates, cup interactions, position or annotation of which cup
holds the juvenile, and none of the behavioural, tracing or RNAscope data of Figures 1 to 5 and 8.

| Claim | Tested here |
|---|---|
| fig6df-epsp-ipsp-at-rest | tested |
| fig6gi-firing-responses | tested |
| fig6j-epsp-amplitude-region | tested |
| fig6k-ipsp-amplitude-region | tested |
| fig6-s1de-monosynaptic | blocked: the TTX/4AP sweeps are not marked in the files |
| fig6no-cea-clusters | tested |
| fig6-s1h-cea-cluster-properties | tested |
| fig6pq-blp-clusters | tested |
| fig6-s1i-blp-cluster-properties | tested |
| fig6-s1jk-cluster-voltage-area | tested |
| fig6-s1ln-combined-clusters | tested |
| fig7-s1ef-red-vs-blue | tested |
| fig7-s1hi-red-during-blue | tested |
| fig7e, fig7gh, fig7j, fig7i, fig7m, fig7-s1no, fig7-s1l, fig7-s2 (in vivo imaging) | blocked: no pose, cup interactions or cup sides in the dataset; cell counts are shown in the data check |
| Figures 1 to 5 and 8 (17 claims: tracing, behaviour, RNAscope) | blocked: not in the dataset |

Loading and measures are in `drn_ephys.py`, which says where each parameter comes from. The
notebook streams every file; it takes about 15 minutes, most of it reading the patch-clamp sweeps.

In [ ]:
import collections, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import h5py, remfile, fsspec, av
from scipy import stats
from scipy.cluster.hierarchy import linkage, fcluster
from statsmodels.stats.anova import AnovaRM
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from concurrent.futures import ThreadPoolExecutor
import drn_ephys as de

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 200, "display.max_columns", 30, "display.max_rows", 200)
print(de.DANDISET_ID, de.DANDISET_VERSION)

## Loading

The file list of the pinned version, the metadata of every imaging file and the length of every behaviour video, and every patch-clamp cell, reduced to the sweeps the analysis uses as it loads.

In [ ]:
ASSETS = de.assets()
icephys = [a for a in ASSETS if a[0].endswith("_icephys.nwb")]
ophys = [a for a in ASSETS if a[0].endswith("image+ophys.nwb")]
client_assets = {a.path: a for a in de.DandiAPIClient().get_dandiset(de.DANDISET_ID, de.DANDISET_VERSION).get_assets()}
videos = {p: client_assets[p].get_content_url(follow_redirects=1, strip_query=True)
          for p in client_assets if p.endswith(".mp4")}
print(len(ASSETS), "NWB files:", len(icephys), "patch-clamp,", len(ophys), "imaging;", len(videos), "videos")

def ophys_meta(path_url):
    path, url = path_url
    with h5py.File(remfile.File(url), "r") as f:
        r = f["processing/ophys/Fluorescence/RoiResponseSeries"]
        g = f["general"]
        return {
            "path": path, "subject": g["subject/subject_id"][()].decode(),
            "subject_description": g["subject/description"][()].decode(),
            "session": f["session_description"][()].decode(),
            "n_rois": r["data"].shape[1], "n_frames": r["data"].shape[0],
            "rate_hz": float(r["starting_time"].attrs["rate"]),
            "video": f["acquisition/BehaviorVideo/external_file"][0].decode(),
            "virus": g["virus"][()].decode(), "stimulus": g["stimulus"][()].decode(),
            "experiment_description": g["experiment_description"][()].decode(),
            "processing_entries": sorted(f["processing/ophys"].keys()),
            "acquisition_entries": sorted(f["acquisition"].keys()),
        }

def video_meta(path):
    with fsspec.open(videos[path], "rb", block_size=2**20) as fh:
        c = av.open(fh)
        s = c.streams.video[0]
        return {"video": path.split("/", 1)[1], "video_s": c.duration / 1e6, "video_frames": s.frames,
                "video_fps": float(s.average_rate)}

with ThreadPoolExecutor(12) as ex:
    OPHYS = pd.DataFrame(list(ex.map(ophys_meta, ophys)))
    VID = pd.DataFrame(list(ex.map(video_meta, sorted(videos))))
OPHYS = OPHYS.merge(VID, on="video", how="left")
print("imaging metadata read")

In [ ]:
CELLS = de.load_cells(icephys, workers=24)
print(len(CELLS), "cells loaded")

## Data check

In [ ]:
# --- patch clamp: what was loaded
cells = pd.DataFrame([{
    "path": c["path"], "subject": c["subject"], "cell": c["cell_id"], "region": c["region"],
    "targeted": c["targeted"], "opsin": "ChR2" if "ChR2" in c["virus"] else ("ChrimsonR" if "Chrimson" in c["virus"] else c["virus"]),
    **{f"n_{k}": len(v) for k, v in c["sweeps"].items()}, "dropped_sweeps": c["dropped"],
    "rates": sorted(c["rates"]),
} for c in CELLS]).fillna(0)
for col in ["n_rest", "n_firing", "n_steps", "n_seal", "n_red", "n_blue03", "n_blue02", "n_red_contblue"]:
    if col not in cells: cells[col] = 0
print("cells by opsin and region (region from DandiIcephysMetadata/inferred_layer):")
print(pd.crosstab(cells.opsin, cells.region, margins=True))
print("\ninferred_layer equals targeted_layer in", (cells.region == cells.targeted).sum(), "of", len(cells), "cells")
print("cell name letter by region:")
print(pd.crosstab(cells.cell.str[0], cells.region))
print("\nsampling rates of the sweeps used (Hz):", sorted({r for rs in cells.rates for r in rs}))
print("subjects:", cells.subject.nunique(), sorted(cells.subject.unique()))
print("\ncells with each kind of sweep, by region (ChR2):")
chr2 = cells[cells.opsin == "ChR2"]
print(pd.DataFrame({k: (chr2[f"n_{k}"] > 0).groupby(chr2.region).sum() for k in ["rest", "firing", "steps", "seal"]}))
print("\nChrimsonR cells with each light condition:")
chrim = cells[cells.opsin == "ChrimsonR"]
print({k: int((chrim[f"n_{k}"] > 0).sum()) for k in ["red", "blue03", "blue02", "red_contblue", "steps"]})
print("\nsweeps not used (other protocols, voltage clamp), total:", int(cells.dropped_sweeps.sum()))

# protocols present, and whether any sweep names TTX or 4-AP
descs = collections.Counter(d for c in CELLS for d in c["descriptions"])
print("\ndistinct sweep descriptions:", len(descs))
print("descriptions naming TTX, 4AP or 4-AP:", [d for d in descs if any(k in d.upper() for k in ("TTX", "4AP", "4-AP"))])
print("session_description values are not loaded per sweep; files without light sweeps at 30 s:",
      int(((chr2.n_rest + chr2.n_firing) == 0).sum()), "ChR2 cells")

# duplicates: identical first 5000 samples of any sweep in two files
owners = collections.defaultdict(set)
for c in CELLS:
    for h in c["hashes"]:
        owners[h].add(c["path"])
dups = [sorted(v) for v in owners.values() if len(v) > 1]
print("\nsweeps found in more than one file:", len(dups))

In [ ]:
# --- patch clamp: units and ranges of what the analysis uses
def summary(x):
    x = np.asarray(x, float)
    return f"min {np.nanmin(x):.2f}, median {np.nanmedian(x):.2f}, max {np.nanmax(x):.2f} (n={np.isfinite(x).sum()})"

rest_base, hold_rest, hold_fire = [], [], []
for c in CELLS:
    for s in c["sweeps"].get("rest", []):
        hold_rest.append(s["hold_pA"])
    for s in c["sweeps"].get("firing", []):
        hold_fire.append(s["hold_pA"])
    if "rest" in c["sweeps"]:
        rest_base.append((c["path"], np.median(de.average(c["sweeps"]["rest"])[:40000])))
print("baseline membrane potential of the rest sweeps, per cell (mV):", summary([b for _, b in rest_base]))
print("  cells above -40 mV:", [(p.split("/")[1][-30:], round(b, 1)) for p, b in rest_base if b > -40])
odd = sorted({c["path"].split("/")[1] for c in CELLS for s in c["sweeps"].get("rest", []) if abs(s["hold_pA"]) > 1e4})
print("ch-1 (labelled volts) during rest sweeps, read as current (pA):", summary([h for h in hold_rest if abs(h) < 1e4]))
print("  cells whose ch-1 is outside +-10 nA in rest sweeps:", odd)
print("ch-1 during firing sweeps (positive holding current in the description), pA:", summary(hold_fire))
steps_i = [np.median(s["i"][2000:11000]) - np.median(s["i"][:500]) for c in CELLS for s in c["sweeps"].get("steps", [])]
print("current steps read from ch-1 (pA):", summary(steps_i))
caps = [de.cell_capacitance_pf(c["sweeps"]["seal"]) for c in CELLS if "seal" in c["sweeps"]]
print("cells whose every Q-method fit failed (0 pF or less):", int(np.isnan(caps).sum()), "of", len(caps))
seal_dv = [np.ptp(np.percentile(c["sweeps"]["seal"][0]["v"], [10, 90])) for c in CELLS if "seal" in c["sweeps"]]
print("seal test voltage step (ch-1, mV, 10th to 90th percentile):", summary(seal_dv))
print("capacitance from the seal test (pF):", summary(caps))

The files do not record when the light was on: the stimulus series of every light sweep are zero.
The helper takes the first pulse 5.468 s into each 30 s light sweep, where the average response
of the responding cells starts. The check below averages the rest sweeps of every ChR2 cell and
compares the deflection at that time with the same measure 6 s later, after the 5.5 s response
window, where no light is given.

In [ ]:
def clock_check(kind="rest", k=3):
    rows = []
    for c in CELLS:
        if kind not in c["sweeps"] or "ChR2" not in c["virus"]:
            continue
        avg = de.average(c["sweeps"][kind])
        base, resp = de.light_window(avg, de.LIGHT_ONSET_S)
        sd = base.std()
        early = resp[: int(0.2 * de.FS)] - base.mean()
        base2, resp2 = de.light_window(avg, de.LIGHT_ONSET_S + 6)
        late = resp2[: int(0.2 * de.FS)] - base2.mean()
        rows.append({"true": np.abs(early).max() / sd, "shifted": np.abs(late).max() / base2.std()})
    return pd.DataFrame(rows)

cc = clock_check()
print("max |deflection| in the first 200 ms, in baseline SDs, ChR2 rest averages (n=%d)" % len(cc))
print(" at the assumed onset: median %.1f, cells above 10 SD: %d" % (cc["true"].median(), (cc["true"] > 10).sum()))
print(" 6 s later:           median %.1f, cells above 10 SD: %d" % (cc.shifted.median(), (cc.shifted > 10).sum()))

# onset of the grand average of responding cells
from scipy.signal import find_peaks
segs = []
for c in CELLS:
    if "rest" in c["sweeps"] and "ChR2" in c["virus"]:
        avg = de.average(c["sweeps"]["rest"])
        base, _ = de.light_window(avg)
        seg = avg[int(5.3 * de.FS):int(5.8 * de.FS)] - base.mean()
        if seg.max() > 10 * base.std():
            segs.append(seg / seg.max())
grand = np.mean(segs, axis=0)
t = 5.3 + np.arange(len(grand)) / de.FS
onset = t[np.argmax(grand > 0.1)]
print(f"grand average of {len(segs)} cells with an EPSP above 10 SD crosses 10% of its peak at {onset:.4f} s")
dg = np.gradient(np.convolve(grand, np.ones(10) / 10, mode="same"))
pk, _ = find_peaks(dg, distance=int(0.02 * de.FS), prominence=dg.max() * 0.05)
pk = t[pk][t[pk] >= onset - 0.005][:8]
print("maxima of its rate of rise from the onset on (s):", np.round(pk, 4), "; intervals (ms):", np.round(np.diff(pk) * 1e3, 1))
fig, ax = plt.subplots(figsize=(8, 2.5))
ax.plot(t, grand); ax.axvline(de.LIGHT_ONSET_S, color="r", lw=.5)
for i in range(8): ax.axvline(de.LIGHT_ONSET_S + i / 30, color="r", lw=.3, ls=":")
ax.set_xlabel("time in sweep (s)"); ax.set_ylabel("normalized Vm"); ax.set_title("grand average, red lines: assumed pulses")
plt.show()

In [ ]:
# --- imaging: what is there, against the paper's counts
OPHYS["condition"] = np.select(
    [OPHYS.session.str.contains("Stim ON"), OPHYS.session.str.startswith("Group housed"), OPHYS.session.str.startswith("24hr")],
    ["GH on", "GH off", "SI off"], "?")
OPHYS["group"] = np.where(OPHYS.subject_description.str.contains("chrimson"), "ChrimsonR", "TdTomato")
print(OPHYS.groupby(["group", "condition"]).agg(mice=("subject", "nunique"), sessions=("path", "size"), rois=("n_rois", "sum")))
print("\nROIs per mouse and condition:")
print(OPHYS.pivot_table(index=["group", "subject"], columns="condition", values="n_rois", aggfunc="sum"))
print("\npaper: ChrimsonR 429, 441 and 484 cells (GH off, GH on, SI off); TdTomato 39, 61 and 76")
print("\nimaging frames:", summary(OPHYS.n_frames), "; stated rate:", OPHYS.rate_hz.unique(), "Hz")
print("behaviour video length (s):", summary(OPHYS.video_s), "; video fps:", OPHYS.video_fps.unique())
OPHYS["imaging_s_at_stated_rate"] = OPHYS.n_frames / OPHYS.rate_hz
OPHYS["imaging_s_at_20hz"] = OPHYS.n_frames / 20.0
print("imaging length at the stated 25 Hz (s):", summary(OPHYS.imaging_s_at_stated_rate))
print("imaging length at the Methods' 20 frames/s (s):", summary(OPHYS.imaging_s_at_20hz))
print("video minus imaging at 20 Hz (s):", summary(OPHYS.video_s - OPHYS.imaging_s_at_20hz))
off = OPHYS[(OPHYS.video_s - OPHYS.imaging_s_at_20hz).abs() > 5]
print("sessions where they differ by more than 5 s:", off[["subject", "condition", "video_s", "imaging_s_at_20hz"]].round(1).to_dict("records"))
print("\nprocessing/ophys holds:", OPHYS.processing_entries.map(tuple).unique())
print("acquisition holds:", OPHYS.acquisition_entries.map(tuple).unique())
for col in ["virus", "stimulus", "experiment_description"]:
    print(f"{col}:", OPHYS[col].unique())

Checks of the description against the data:

- Sampling at 10 kHz in every sweep used, as in the Methods.
- In current clamp ch-1 is labelled volts, but its values are tens of pA, positive in the sweeps
  whose description names a positive holding current, and follow the 1 s steps from -120 to 260 pA:
  ch-1 is the injected current. The seal test is stored as an `IZeroClampSeries` but its ch-1 is a
  square voltage command of about 5 mV and its ch-0 a current: it is voltage clamp.
- The light time is not in the files. The assumed onset is where the responses start, and the
  following maxima of the grand average fall 33 ms apart, as 30 Hz pulses would; the response at
  the assumed onset is far above the same measure 6 s later. This is consistent with the
  description of 8 pulses at 30 Hz but does not prove the onset to the millisecond.
- No sweep names TTX or 4-AP, so the TTX/4AP recordings of Figure 6 supplement 1D-E cannot be
  identified.
- Imaging: the traces are stated at 25 Hz, but 19,211 frames at 25 Hz last 768 s while the
  behaviour videos last 961 s (median; one lasts 800 s); at the Methods' 20 frames/s the imaging
  lasts 961 s, within 4 s of the video in every session but that one. The data support 20 Hz, and the stated rate is wrong.
- The number of ROIs is about twice the paper's cell counts in every condition (above), and the
  files do not mark which ROIs the paper kept.

## fig6df-epsp-ipsp-at-rest

> At the resting membrane potential, photostimulation of DRN-DAT terminals evoked both excitatory
> and inhibitory post-synaptic potentials in downstream cells of the BNST, CeA and BLP (19, 36 and
> 48 cells recorded).

Per ChR2 cell, the average of its rest sweeps (30 s light sweeps with no or negative holding
current; Methods). Peak and trough within 0-5.5 s of the first pulse against the 5 s baseline,
detected beyond 2 or 3 SD of that baseline (the criterion's readings); 5 SD is shown as a further
variant. Cells recorded only with other protocols (20 s or 60 s sweeps, or light at 10 s) are
left out because their light time cannot be fixed; their number is printed.

In [ ]:
REGIONS = ["BNST", "CeA", "BLP"]
EXPECTED_REST = {"BNST": 19, "CeA": 36, "BLP": 48}
rest = []
for c in CELLS:
    if "ChR2" not in c["virus"] or "rest" not in c["sweeps"]:
        continue
    avg = de.average(c["sweeps"]["rest"])
    row = {"path": c["path"], "subject": c["subject"], "cell": c["cell_id"], "region": c["region"],
           "n_sweeps": len(c["sweeps"]["rest"])}
    for k in (2, 3, 5):
        e = de.evoked(avg, k)
        row[f"class_k{k}"] = de.response_class(e)
        row[f"epsp_k{k}"], row[f"ipsp_k{k}"] = e["epsp"], e["ipsp"]
    row.update({kk: e[kk] for kk in ("baseline_mV", "sd_mV", "peak_mV", "trough_mV", "area_mVs")})
    rest.append(row)
REST = pd.DataFrame(rest)
no_rest = cells[(cells.opsin == "ChR2") & (cells.n_rest == 0)]
print("ChR2 cells without 30 s rest sweeps (left out):", len(no_rest), dict(no_rest.region.value_counts()))
print("cells per region:", {r: int((REST.region == r).sum()) for r in REGIONS}, "expected", EXPECTED_REST)
for k in (2, 3, 5):
    tab = pd.crosstab(REST.region, REST[f"class_k{k}"]).reindex(REGIONS).fillna(0).astype(int)
    print(f"\nreading k={k} SD:"); print(tab)
    print("  percent:"); print((100 * tab.div(tab.sum(axis=1), axis=0)).round(1))
    ok = all(REST[(REST.region == r)][f"epsp_k{k}"].any() for r in REGIONS) and REST[f"ipsp_k{k}"].any()
    print(f"  every region has a cell with an EPSP and some cell has an IPSP: {ok}")
print("\nby subject (k=3):")
print(pd.crosstab([REST.region, REST.subject], REST.class_k3))
print("\npeak (mV):", summary(REST.peak_mV), "\ntrough (mV):", summary(REST.trough_mV), "\nbaseline SD (mV):", summary(REST.sd_mV))

**Result: reproduced.** Under both readings every region has cells with an EPSP and cells with an IPSP: at 2 SD the
BNST, CeA and BLP have 14, 39 and 54 cells with an EPSP and 10, 27 and 48 with an IPSP; at 3 SD 12, 37 and 54, and
6, 11 and 34. The dataset holds 15, 40 and 56 ChR2 cells with 30 s rest sweeps where the paper reports 19, 36 and 48,
so the counts do not match; 19 ChR2 cells recorded only with other light protocols are left out. The detection rule
the criterion names classes most cells as mix (65 to 82% at 2 SD); at 5 SD most cells are excitation only.

## fig6gi-firing-responses

> In cells made to fire by constant current, photostimulation increased firing in all BNST cells
> (5 of 5), while CeA (20 cells) and BLP (17 cells) cells showed either an increase or a decrease in
> firing.

Firing sweeps are the 30 s light sweeps with a positive holding current (Methods). Spikes are
upward crossings of 0 mV. IEI before (5 s, or 5.5 s as the Figure 6 supplement 1G variant) and 5 s
after the first pulse; readings (a) mean of all IEIs pooled over sweeps, (b) mean over sweeps of
each sweep's mean IEI.

In [ ]:
EXPECTED_FIRE = {"BNST": 5, "CeA": 20, "BLP": 17}
fire = []
for c in CELLS:
    if "ChR2" not in c["virus"] or "firing" not in c["sweeps"]:
        continue
    row = {"path": c["path"], "subject": c["subject"], "cell": c["cell_id"], "region": c["region"],
           "n_sweeps": len(c["sweeps"]["firing"])}
    for before in (5.0, 5.5):
        r = de.iei_change(c["sweeps"]["firing"], before_s=before)
        for rd in ("a", "b"):
            row[f"{rd}_{before}_before"], row[f"{rd}_{before}_after"] = r[f"{rd}_before"], r[f"{rd}_after"]
            row[f"label_{rd}_{before}"] = de.iei_label(r[f"{rd}_before"], r[f"{rd}_after"])
    fire.append(row)
FIRE = pd.DataFrame(fire)
print("cells per region:", {r: int((FIRE.region == r).sum()) for r in REGIONS}, "expected", EXPECTED_FIRE)
for before in (5.0, 5.5):
    for rd in ("a", "b"):
        col = f"label_{rd}_{before}"
        tab = pd.crosstab(FIRE.region, FIRE[col].fillna("no IEI")).reindex(REGIONS).fillna(0).astype(int)
        lab_bnst = FIRE[(FIRE.region == "BNST") & FIRE[col].notna()][col]
        bnst_all = len(lab_bnst) > 0 and (lab_bnst == "excitation").all()
        both = all(set(FIRE[FIRE.region == r][col]) >= {"excitation", "inhibition"} for r in ("CeA", "BLP"))
        print(f"\nreading ({rd}), baseline {before} s:"); print(tab)
        print(f"  all BNST cells with an IEI excited ({len(lab_bnst)} cells): {bnst_all}; both classes in CeA and in BLP: {both}")
FIRE["ratio_a5"] = FIRE["a_5.0_after"] / FIRE["a_5.0_before"]
print("\nper cell, IEI after/before, reading (a), 5 s:")
print(FIRE[["region", "subject", "cell", "n_sweeps", "a_5.0_before", "a_5.0_after", "ratio_a5", "label_a_5.0"]].round(3).sort_values("region").to_string())

**Result: partially reproduced.** In every reading the BNST cells that could be classed are excited (2 of 2), and
both excitation and inhibition occur in the CeA (8 and 2) and the BLP (4 to 6 and 7 to 9). The dataset holds 3, 13
and 13 cells with firing sweeps, where the paper reports 5, 20 and 17, and one BNST cell (B7, one sweep) has no
spikes before the light, so the paper's 5 of 5 BNST cells cannot be checked.

## fig6j-epsp-amplitude-region

> The peak amplitude of the optically evoked EPSP differed between regions, larger in the CeA than in
> the BLP.

EPSP peak of the average rest trace, one value per cell with an EPSP; cells (a) excitation and mix,
(b) excitation only; under each detection reading of fig6df (2 and 3 SD). Kruskal-Wallis, then
Dunn's test (Prism's adjustment).

In [ ]:
def fig6j(k, which):
    cls = REST[f"class_k{k}"]
    sel = REST[cls.isin(["excitation", "mix"] if which == "a" else ["excitation"])]
    g = {r: sel[sel.region == r].peak_mV.values for r in REGIONS}
    if min(len(v) for v in g.values()) < 2:
        return {"k": k, "cells": which, **{f"n_{r}": len(g[r]) for r in REGIONS}, "H": np.nan, "p": np.nan}
    H, p = stats.kruskal(*g.values())
    dn = de.dunn(g)
    return {"k": k, "cells": which, **{f"n_{r}": len(g[r]) for r in REGIONS},
            **{f"median_{r}": np.median(g[r]) for r in REGIONS}, "H": H, "p": p,
            "dunn_CeA_BLP": dn[("CeA", "BLP")], "dunn_BNST_CeA": dn[("BNST", "CeA")], "dunn_BNST_BLP": dn[("BNST", "BLP")]}
J = pd.DataFrame([fig6j(k, w) for k in (2, 3) for w in ("a", "b")])
J["holds"] = (J.p < 0.05) & (J.dunn_CeA_BLP < 0.05) & (J.median_CeA > J.median_BLP)
print(J.round(4).to_string())
print("\npaper: H = 6.790, p = 0.0335; Dunn CeA vs BLP p = 0.0378")

**Result: reproduced, under reading (a).** With all cells with an EPSP (excitation and mix), Kruskal-Wallis gives
p = 0.031 (2 SD) and 0.0066 (3 SD), Dunn's test CeA against BLP p = 0.042 and 0.015, and the CeA median (2.6 and 2.8 mV)
exceeds the BLP median (1.4 mV). With excitation-only cells (reading b) no test is significant (p = 0.76 and 0.55).
The readings disagree; reading (a) is adopted, as the EPSP of Figure 6J is measured in every cell that has one.

## fig6k-ipsp-amplitude-region

> The trough amplitude of the optically evoked IPSP differed between regions, deeper in the CeA than
> in the BLP.

IPSP trough of the average rest trace; cells (a) inhibition and mix, (b) inhibition only; detection
at 2 or 3 SD. One-way ANOVA, then Tukey's test and Holm-Sidak (the post hoc test is not named).

In [ ]:
def fig6k(k, which):
    cls = REST[f"class_k{k}"]
    sel = REST[cls.isin(["inhibition", "mix"] if which == "a" else ["inhibition"])]
    g = {r: sel[sel.region == r].trough_mV.values for r in REGIONS if (sel.region == r).sum() > 0}
    out = {"k": k, "cells": which, "n_total": len(sel), **{f"n_{r}": int((sel.region == r).sum()) for r in REGIONS},
           **{f"mean_{r}": sel[sel.region == r].trough_mV.mean() for r in REGIONS}}
    usable = {r: v for r, v in g.items() if len(v) >= 2}
    if len(usable) < 2 or "CeA" not in usable or "BLP" not in usable:
        return {**out, "F": np.nan, "p": np.nan}
    F, p = stats.f_oneway(*usable.values())
    tk = pairwise_tukeyhsd(np.concatenate(list(usable.values())), np.concatenate([[r] * len(v) for r, v in usable.items()]))
    res = pd.DataFrame(tk._results_table.data[1:], columns=tk._results_table.data[0])
    t_cea_blp = float(res[((res.group1 == "BLP") & (res.group2 == "CeA")) | ((res.group1 == "CeA") & (res.group2 == "BLP"))]["p-adj"].iloc[0])
    hs = de.holm_sidak(usable)
    hs_cea_blp = hs.get(("CeA", "BLP"), hs.get(("BLP", "CeA")))
    return {**out, "groups_in_test": len(usable), "df": f"{len(usable) - 1},{sum(len(v) for v in usable.values()) - len(usable)}",
            "F": F, "p": p, "tukey_CeA_BLP": t_cea_blp, "holm_sidak_CeA_BLP": hs_cea_blp}
K = pd.DataFrame([fig6k(k, w) for k in (2, 3) for w in ("a", "b")])
K["holds_tukey"] = (K.p < 0.05) & (K.tukey_CeA_BLP < 0.05) & (K.mean_CeA < K.mean_BLP)
K["holds_holm_sidak"] = (K.p < 0.05) & (K.holm_sidak_CeA_BLP < 0.05) & (K.mean_CeA < K.mean_BLP)
K["n_within_34_pm_3.4"] = (K.n_total - 34).abs() <= 3.4
print(K.round(4).to_string())
print("\npaper: F(2,31) = 8.150, p = 0.0014, CeA vs BLP p = 0.0014; 34 cells")

**Result: not reproduced.** No reading gives the paper's result. With inhibition and mix cells the ANOVA gives
p = 0.69 (2 SD, 85 cells) and p = 0.026 (3 SD, 51 cells), and in the second CeA against BLP is not significant (Tukey
p = 0.16, Holm-Sidak p = 0.13) though the CeA mean trough is deeper (-0.80 against -0.49 mV). With inhibition-only
cells there are 3 or 4 cells, too few for the test. The paper's test has 34 cells; the detection rule finds 51 or 85.

## fig6-s1de-monosynaptic (blocked)

> The optically evoked EPSP and the slow component of the IPSP persisted after TTX/4AP, so the
> responses were typically monosynaptic.

No sweep description, session description or other field names TTX or 4-AP (data check), so the
sweeps recorded after TTX/4AP cannot be told from those before. Blocked, `metadata_insufficient`.

## Baseline properties for the clustering claims

Per ChR2 cell from its 1 s current steps (Methods): voltage sag at -120 pA (steady state the last
100 ms, or 200 ms, of the step), ramp ratio at the largest subthreshold step (absolute Vm as
written, or deflections from the pre-step Vm), firing delay at rheobase, and the maximum
instantaneous frequency in the first 100 ms of the depolarizing steps (0 Hz when no two spikes fall
in that time). Capacitance by the Q-method from the 5 mV seal-test pulses, the only voltage-clamp
test pulses in the files (the Methods use a 5 mV, 250 ms step). Some BLP cells were also stepped
from 220 pA up to 600 pA; those steps are used for rheobase.

Response label per cell (Methods): from the IEI when the cell has firing sweeps, otherwise
excitation for an EPSP only, inhibition for an IPSP only, the sign of the 0-5.5 s voltage area for
a mix, and none otherwise.

In [ ]:
feat_rows = []
for c in CELLS:
    if "ChR2" not in c["virus"]:
        continue
    row = {"path": c["path"], "subject": c["subject"], "cell": c["cell_id"], "region": c["region"]}
    if "steps" in c["sweeps"]:
        for ss in (100, 200):
            f = de.step_features(c["sweeps"]["steps"], sag_ss_ms=ss) or {}
            row[f"sag_{ss}"] = f.get("sag_mV", np.nan)
        row.update({"ramp_abs": f.get("ramp_ratio", np.nan), "ramp_defl": f.get("ramp_ratio_deflection", np.nan),
                    "delay": f.get("firing_delay_ms", np.nan), "maxfreq": f.get("max_freq_hz", np.nan),
                    "step_min": f.get("min_pA", np.nan), "step_max": f.get("max_pA", np.nan)})
    if "seal" in c["sweeps"]:
        row["cap"] = de.cell_capacitance_pf(c["sweeps"]["seal"])
    feat_rows.append(row)
FEAT = pd.DataFrame(feat_rows)
FEAT = FEAT.merge(REST[["path", "class_k2", "class_k3", "peak_mV", "trough_mV", "area_mVs", "epsp_k2", "epsp_k3"]], on="path", how="left")
FEAT = FEAT.merge(FIRE[["path", "label_a_5.0", "label_b_5.0"]], on="path", how="left")

def label(row, k, iei):
    lab = row.get(f"label_{iei}_5.0")
    if isinstance(lab, str):
        return lab
    cls = row.get(f"class_k{k}")
    if cls == "mix":
        return "excitation" if row["area_mVs"] > 0 else "inhibition"
    return {"excitation": "excitation", "inhibition": "inhibition", "none": "none"}.get(cls, "no data")
for k in (2, 3):
    for iei in ("a", "b"):
        FEAT[f"resp_k{k}_{iei}"] = FEAT.apply(lambda r: label(r, k, iei), axis=1)
print("ChR2 cells with current steps:", int(FEAT.step_min.notna().sum()), "; with each feature, by region:")
print(FEAT.groupby("region")[["sag_100", "ramp_abs", "delay", "maxfreq", "cap"]].count())
for col in ["sag_100", "sag_200", "ramp_abs", "ramp_defl", "delay", "maxfreq", "cap"]:
    print(f"{col}: {summary(FEAT[col])}")

In [ ]:
CEA_FEATS = ["ramp", "maxfreq", "delay", "sag"]
BLP_FEATS = ["cap", "maxfreq", "delay", "sag"]

def cluster(df, cols):
    X = df[cols].to_numpy(float)
    X = (X - X.min(0)) / (X.max(0) - X.min(0))
    return fcluster(linkage(X, method="ward", metric="euclidean"), 2, criterion="maxclust")

def frame_for(region, ramp, sag, feats):
    df = FEAT[FEAT.region.isin([region] if isinstance(region, str) else region)].copy()
    df["ramp"], df["sag"] = df[ramp], df[sag]
    return df.dropna(subset=feats).reset_index(drop=True)

## fig6no-cea-clusters

> Hierarchical clustering of CeA cells on four baseline properties gave two clusters, late-firing
> cells (cluster 1) that were excited by DRN-DAT input and regular-firing cells (cluster 2) that were
> mostly inhibited.

Max-min normalized ramp ratio, maximum instantaneous frequency, firing delay and sag; Ward linkage
on Euclidean distance; two clusters. The cluster with the longer mean firing delay is the late-firing
cluster. Every reading of the criterion is run: ramp ratio absolute or from deflections, sag
steady state 100 or 200 ms, and the detection (2 or 3 SD) and IEI readings behind the labels.

In [ ]:
CEA_RUNS = []
for ramp in ("ramp_abs", "ramp_defl"):
    for sag in ("sag_100", "sag_200"):
        df = frame_for("CeA", ramp, sag, CEA_FEATS)
        df["cl"] = cluster(df, CEA_FEATS)
        late = df.groupby("cl").delay.mean().idxmax()
        df["late"] = df.cl == late
        for k in (2, 3):
            for iei in ("a", "b"):
                lab = df[f"resp_k{k}_{iei}"]
                L, R = lab[df.late], lab[~df.late]
                CEA_RUNS.append({"ramp": ramp, "sag": sag, "k": k, "iei": iei, "n": len(df),
                                 "n_late": len(L), "late_excited": (L == "excitation").sum(), "late_inhibited": (L == "inhibition").sum(),
                                 "late_none_or_nodata": L.isin(["none", "no data"]).sum(),
                                 "n_regular": len(R), "regular_excited": (R == "excitation").sum(),
                                 "regular_inhibited": (R == "inhibition").sum(),
                                 "regular_none_or_nodata": R.isin(["none", "no data"]).sum(),
                                 "holds": (L == "excitation").all() and (R == "inhibition").sum() > len(R) / 2})
        if ramp == "ramp_abs" and sag == "sag_100":
            CEA_MAIN = df.copy()
CEA_RUNS = pd.DataFrame(CEA_RUNS)
print(CEA_RUNS.to_string())
print("\ncells clustered:", CEA_RUNS.n.unique(), "(expected 26); subjects:", CEA_MAIN.subject.nunique())
print("\nfirst reading, cluster means of the raw features:")
print(CEA_MAIN.groupby("late")[CEA_FEATS].mean().round(3))
print("\nlabels by cluster and subject, first reading (k=3, IEI a):")
print(pd.crosstab([CEA_MAIN.late, CEA_MAIN.subject], CEA_MAIN.resp_k3_a))

**Result: partially reproduced.** Every reading gives two clusters of 27 cells (the paper has 26), a late-firing
cluster (mean firing delay 415 against 168 ms) and a regular-firing one. The late-firing cluster is mostly excited
(11 to 13 of 17 or 18 cells) but holds 2 or 3 inhibited cells, so not every cell is excited as the criterion requires;
5 of the 9 or 10 regular-firing cells are inhibited, more than half. The readings agree.

## fig6-s1h-cea-cluster-properties

> The two CeA clusters differed in all four input properties (ramp ratio, maximum instantaneous
> frequency, firing delay and voltage sag).

Unpaired two-sided t-tests between the clusters of fig6no-cea-clusters on the raw features, under
each clustering reading.

In [ ]:
rows = []
for ramp in ("ramp_abs", "ramp_defl"):
    for sag in ("sag_100", "sag_200"):
        df = frame_for("CeA", ramp, sag, CEA_FEATS)
        df["cl"] = cluster(df, CEA_FEATS)
        a, b = df[df.cl == 1], df[df.cl == 2]
        row = {"ramp": ramp, "sag": sag, "n1": len(a), "n2": len(b)}
        for f in CEA_FEATS:
            t, p = stats.ttest_ind(a[f], b[f])
            row[f"t_{f}"], row[f"p_{f}"] = t, p
        row["all_p<0.05"] = all(row[f"p_{f}"] < 0.05 for f in CEA_FEATS)
        rows.append(row)
H = pd.DataFrame(rows)
print(H.round(5).to_string())
print("\npaper: t24 = 3.502 (ramp), 4.698 (max freq), 5.050 (delay), 3.983 (sag)")

**Result: reproduced.** Under every reading the two CeA clusters differ in all four properties (p from 0.029 to
below 0.0001).

## fig6pq-blp-clusters

> BLP cells clustered into putative pyramidal neurons (cluster 1) and putative GABAergic interneurons
> (cluster 2); 93% of the putative pyramidal neurons were inhibited by DRN-DAT input and 62% of the
> putative interneurons were excited.

As for the CeA with capacitance in place of the ramp ratio. The pyramidal cluster is the one with
the larger mean capacitance; whether it also has the lower maximum frequency is printed.
Percentages over all cells of a cluster.

In [ ]:
BLP_RUNS = []
for sag in ("sag_100", "sag_200"):
    df = frame_for("BLP", "ramp_abs", sag, BLP_FEATS)
    df["cl"] = cluster(df, BLP_FEATS)
    means = df.groupby("cl")[["cap", "maxfreq"]].mean()
    pyr = means.cap.idxmax()
    df["pyr"] = df.cl == pyr
    for k in (2, 3):
        for iei in ("a", "b"):
            lab = df[f"resp_k{k}_{iei}"]
            P, I = lab[df.pyr], lab[~df.pyr]
            pi, ie = 100 * (P == "inhibition").mean(), 100 * (I == "excitation").mean()
            BLP_RUNS.append({"sag": sag, "k": k, "iei": iei, "n": len(df), "n_pyr": len(P), "n_int": len(I),
                             "pyr_has_lower_maxfreq": means.maxfreq[pyr] < means.maxfreq.drop(pyr).iloc[0],
                             "pyr_inhibited_%": pi, "pyr_excited_%": 100 * (P == "excitation").mean(),
                             "int_excited_%": ie, "int_inhibited_%": 100 * (I == "inhibition").mean(),
                             "within_tol": abs(pi - 93) <= 9.3 and abs(ie - 62) <= 6.2,
                             "direction": (P == "inhibition").mean() > 0.5 and (I == "excitation").mean() > 0.5})
    if sag == "sag_100":
        BLP_MAIN = df.copy()
BLP_RUNS = pd.DataFrame(BLP_RUNS)
print(BLP_RUNS.round(1).to_string())
print("\ncells clustered:", BLP_RUNS.n.unique(), "(expected 27)")
print("\nfirst reading, cluster means of the raw features:")
print(BLP_MAIN.groupby("pyr")[BLP_FEATS].mean().round(2))
print("\nlabels by cluster (k=3, IEI a):")
print(pd.crosstab(BLP_MAIN.pyr, BLP_MAIN.resp_k3_a))
print("\nby subject:")
print(pd.crosstab([BLP_MAIN.pyr, BLP_MAIN.subject], BLP_MAIN.resp_k3_a))

**Result: partially reproduced.** Every reading gives 28 BLP cells (the paper has 27) in a cluster of larger
capacitance and lower maximum frequency (17 cells, the putative pyramidal neurons) and one of 11. 59% or 65% of the
pyramidal cluster are inhibited (expected 93 ± 9.3%) and 73% of the other cluster are excited (expected 62 ± 6.2%):
the direction holds, the percentages are outside tolerance.

## fig6-s1i-blp-cluster-properties

> The two BLP clusters differed in capacitance, maximum instantaneous frequency, firing delay and
> voltage sag.

In [ ]:
rows = []
for sag in ("sag_100", "sag_200"):
    df = frame_for("BLP", "ramp_abs", sag, BLP_FEATS)
    df["cl"] = cluster(df, BLP_FEATS)
    a, b = df[df.cl == 1], df[df.cl == 2]
    row = {"sag": sag, "n1": len(a), "n2": len(b)}
    for f in BLP_FEATS:
        t, p = stats.ttest_ind(a[f], b[f])
        row[f"t_{f}"], row[f"p_{f}"] = t, p
    row["all_p<0.05"] = all(row[f"p_{f}"] < 0.05 for f in BLP_FEATS)
    rows.append(row)
I_ = pd.DataFrame(rows)
print(I_.round(5).to_string())
print("\npaper: t25 = 4.803 (capacitance), 15.48 (max freq), 2.743 (delay), 2.705 (sag)")

**Result: reproduced.** Under both readings the two BLP clusters differ in capacitance, maximum frequency, firing
delay and sag (p from 0.0032 to below 0.0001).

## fig6-s1jk-cluster-voltage-area

> The total voltage area evoked by DRN-DAT stimulation differed between the two clusters in both the
> CeA and the BLP, and the EPSP peak amplitude differed between the BLP clusters but not between the
> CeA clusters.

Total voltage area 0-5.5 s against the 5 s baseline of the average rest trace (Methods);
Mann-Whitney U for the area, unpaired t-test for the EPSP peak of cells with an EPSP, between the
clusters of each clustering reading.

In [ ]:
rows = []
def jk(df, grp, region, reading):
    out = {"region": region, "reading": reading}
    a, b = df[df[grp]], df[~df[grp]]
    aa, bb = a.area_mVs.dropna(), b.area_mVs.dropna()
    out["n_area"] = f"{len(aa)}+{len(bb)}"
    out["U"], out["p_area"] = stats.mannwhitneyu(aa, bb, alternative="two-sided") if len(aa) and len(bb) else (np.nan, np.nan)
    out["median_area_1"], out["median_area_2"] = aa.median(), bb.median()
    for k in (2, 3):
        ea = a[a[f"epsp_k{k}"] == True].peak_mV
        eb = b[b[f"epsp_k{k}"] == True].peak_mV
        t, p = stats.ttest_ind(ea, eb) if len(ea) > 1 and len(eb) > 1 else (np.nan, np.nan)
        out[f"n_epsp_k{k}"], out[f"t_epsp_k{k}"], out[f"p_epsp_k{k}"] = f"{len(ea)}+{len(eb)}", t, p
    return out
for ramp in ("ramp_abs", "ramp_defl"):
    for sag in ("sag_100", "sag_200"):
        df = frame_for("CeA", ramp, sag, CEA_FEATS)
        df["cl"] = cluster(df, CEA_FEATS)
        df["g"] = df.cl == df.groupby("cl").delay.mean().idxmax()
        rows.append(jk(df, "g", "CeA (late vs regular)", f"{ramp}, {sag}"))
for sag in ("sag_100", "sag_200"):
    df = frame_for("BLP", "ramp_abs", sag, BLP_FEATS)
    df["cl"] = cluster(df, BLP_FEATS)
    df["g"] = df.cl == df.groupby("cl").cap.mean().idxmax()
    rows.append(jk(df, "g", "BLP (pyramidal vs interneuron)", sag))
JK = pd.DataFrame(rows)
print(JK.round(4).to_string())
for k in (2, 3):
    cea, blp = JK[JK.region.str.startswith("CeA")], JK[JK.region.str.startswith("BLP")]
    print(f"k={k}: holds in every reading:",
          bool((cea.p_area < 0.05).all() and (blp.p_area < 0.05).all() and (blp[f"p_epsp_k{k}"] < 0.05).all() and (cea[f"p_epsp_k{k}"] >= 0.05).all()))
print("\npaper: area CeA U = 22, p = 0.0023; BLP U = 29, p = 0.0019; EPSP CeA t17 = 1.40, p = 0.180; BLP t22 = 2.34, p = 0.029")

**Result: reproduced.** Under every reading the total voltage area differs between the CeA clusters (Mann-Whitney
p = 0.0001 to 0.0005) and between the BLP clusters (p = 0.011), the EPSP peak differs between the BLP clusters
(t-test p = 0.014) and not between the CeA clusters (p = 0.42 to 0.83).

## fig6-s1ln-combined-clusters

> Clustering CeA and BLP cells together gave two clusters with a response pattern very similar to
> that of the separate clusterings.

Five features (ramp ratio, capacitance, maximum frequency, firing delay, sag), max-min normalized,
Ward, two clusters, on the CeA and BLP cells with all five.

In [ ]:
FIVE = ["ramp", "cap", "maxfreq", "delay", "sag"]
rows = []
for ramp in ("ramp_abs", "ramp_defl"):
    for sag in ("sag_100", "sag_200"):
        df = frame_for(["CeA", "BLP"], ramp, sag, FIVE)
        df["cl"] = cluster(df, FIVE)
        for k in (2, 3):
            for iei in ("a", "b"):
                lab = df[f"resp_k{k}_{iei}"]
                row = {"ramp": ramp, "sag": sag, "k": k, "iei": iei, "n": len(df)}
                exc_cluster = {}
                for reg in ("CeA", "BLP"):
                    for cl in (1, 2):
                        s = lab[(df.region == reg) & (df.cl == cl)]
                        row[f"{reg}_c{cl}"] = f"{(s == 'excitation').sum()}e/{(s == 'inhibition').sum()}i/{len(s)}"
                        row[f"{reg}_c{cl}_exc_frac"] = (s == "excitation").mean() if len(s) else np.nan
                        row[f"{reg}_c{cl}_inh_frac"] = (s == "inhibition").mean() if len(s) else np.nan
                    e = [cl for cl in (1, 2) if row[f"{reg}_c{cl}_exc_frac"] > 0.5]
                    i = [cl for cl in (1, 2) if row[f"{reg}_c{cl}_inh_frac"] > 0.5]
                    exc_cluster[reg] = e[0] if len(e) == 1 and len(i) == 1 and e != i else None
                row["holds"] = exc_cluster["CeA"] is not None and exc_cluster["CeA"] == exc_cluster["BLP"]
                rows.append(row)
        if ramp == "ramp_abs" and sag == "sag_100":
            COMB = df.copy()
LN = pd.DataFrame(rows)
print(LN[[c for c in LN.columns if not c.endswith("frac")]].to_string())
# agreement with the separate clusterings (first reading)
sep = pd.concat([CEA_MAIN[["path"]].assign(sep=np.where(CEA_MAIN.late, "CeA late", "CeA regular")),
                 BLP_MAIN[["path"]].assign(sep=np.where(BLP_MAIN.pyr, "BLP pyramidal", "BLP interneuron"))])
print("\ncombined cluster against separate cluster (first reading):")
print(pd.crosstab(COMB.merge(sep, on="path").sep, COMB.merge(sep, on="path").cl))

**Result: not reproduced.** In every reading the combined clustering puts the CeA late-firing cells with the BLP
putative pyramidal cells (cluster 2) and the CeA regular-firing cells with the BLP putative interneurons (cluster 1).
The CeA cells of cluster 2 are mostly excited (11 to 14 of 17 or 19) and 5 of the 8 or 10 CeA cells of cluster 1 are
inhibited, while the BLP cells of cluster 1 are mostly excited (6 of 8) and those of cluster 2 mostly inhibited (10 or
11 of 17): the excited cluster is not the same in the two regions. The combined clusters agree with the separate ones
The separate clusters map onto the combined ones, all but one CeA regular-firing cell, in the first reading.

## fig7-s1ef-red-vs-blue

> Ex vivo, red light evoked ChrimsonR-mediated potentials in CeA cells with much larger peak
> amplitude and area than the blue light (0.3 or 0.2 mW) used for imaging.

The ChrimsonR cells' sweeps are named by description: "635nm ... every 30s" (red alone), "...,
+0.3mW blue light" and "..., +0.2mW blue light" (read as the blue light conditions of Figure 7
supplement 1E-F), and "..., +0.3mW cont blue light" (red during constant blue, panels G-I). The
descriptions do not say whether the blue sweeps also carried red pulses; this reading is an
assumption. Amplitude readings: (a) the largest absolute deflection from the 5 s baseline, (b) the
EPSP peak, or the IPSP trough magnitude when only an IPSP is detected (2 or 3 SD). Area readings:
the magnitude of the signed 0-5.5 s area, and the area of the absolute deflection. Repeated measures
one-way ANOVA across the three lights on the cells with all three.

In [ ]:
def psp(avg, k):
    e = de.evoked(avg, k)
    a_abs = max(e["peak_mV"], -e["trough_mV"])
    if e["epsp"]:
        a_b = e["peak_mV"]
    elif e["ipsp"]:
        a_b = -e["trough_mV"]
    else:
        a_b = 0.0
    return {"amp_a": a_abs, f"amp_b_k{k}": a_b, "area_signed": abs(e["area_mVs"]), "area_abs": e["abs_area_mVs"],
            f"detected_k{k}": e["epsp"] or e["ipsp"]}
crow = []
for c in CELLS:
    if "Chrimson" not in c["virus"]:
        continue
    for cond in ("red", "blue03", "blue02", "red_contblue"):
        if cond in c["sweeps"]:
            avg = de.average(c["sweeps"][cond])
            r = {"path": c["path"], "subject": c["subject"], "cell": c["cell_id"], "cond": cond, "n_sweeps": len(c["sweeps"][cond])}
            for k in (2, 3):
                r.update(psp(avg, k))
            crow.append(r)
CHR = pd.DataFrame(crow)
print("ChrimsonR cells per condition:", CHR.groupby("cond").path.nunique().to_dict())
three = CHR[CHR.cond.isin(["red", "blue03", "blue02"])]
complete = three.groupby("path").cond.nunique()
complete = complete[complete == 3].index
T3 = three[three.path.isin(complete)]
print("cells with all three lights:", len(complete), "(paper: 9, from F(2,16))")
EF = []
for m in ["amp_a", "amp_b_k2", "amp_b_k3", "area_signed", "area_abs"]:
    res = AnovaRM(T3, depvar=m, subject="path", within=["cond"]).fit().anova_table
    w = T3.pivot(index="path", columns="cond", values=m)
    EF.append({"measure": m, "F": res["F Value"].iloc[0], "df": f"{res['Num DF'].iloc[0]:.0f},{res['Den DF'].iloc[0]:.0f}",
               "p": res["Pr > F"].iloc[0], **{f"mean_{c}": w[c].mean() for c in ("red", "blue03", "blue02")},
               "red>0.3mW": int((w.red > w.blue03).sum()), "red>0.2mW": int((w.red > w.blue02).sum()), "n": len(w),
               "p_red_vs_0.3 (paired t)": stats.ttest_rel(w.red, w.blue03).pvalue,
               "p_red_vs_0.2 (paired t)": stats.ttest_rel(w.red, w.blue02).pvalue})
EF = pd.DataFrame(EF)
EF["holds"] = (EF.p < 0.05) & (EF["red>0.3mW"] > EF.n / 2) & (EF["red>0.2mW"] > EF.n / 2)
print(EF.round(4).to_string())
print("\nper cell (amplitude reading a, mV):")
print(T3.pivot(index=["subject", "cell"], columns="cond", values="amp_a").round(2))

**Result: reproduced.** In the 9 ChrimsonR cells with all three lights (the paper's 9), red light evokes larger
potentials than either blue light under every reading: amplitude F(2,16) = 25.6, p < 0.0001 (mean 9.9 mV against 1.8
and 0.9 mV), larger in 8 of 9 cells against each blue; area F(2,16) = 11.0 or 7.0, p = 0.001 or 0.0067, larger in 9 of
9. The paper's F values (200.1 and 404.2) are much larger.

## fig7-s1hi-red-during-blue

> Red light still evoked PSPs during constant blue light, though they were significantly smaller than
> with red light alone.

Red alone against red during constant 0.3 mW blue light ("+0.3mW cont blue light"), the same
measures, paired two-sided t-tests on the cells with both.

In [ ]:
two = CHR[CHR.cond.isin(["red", "red_contblue"])]
both = two.groupby("path").cond.nunique()
both = both[both == 2].index
T2 = two[two.path.isin(both)]
print("cells with both:", len(both), "(paper: 12, from t11)")
HI = []
for m in ["amp_a", "amp_b_k2", "amp_b_k3", "area_signed", "area_abs"]:
    w = T2.pivot(index="path", columns="cond", values=m)
    t, p = stats.ttest_rel(w.red, w.red_contblue)
    HI.append({"measure": m, "t": t, "df": len(w) - 1, "p": p, "mean_red": w.red.mean(), "mean_red_during_blue": w.red_contblue.mean(),
               "smaller_during_blue": int((w.red_contblue < w.red).sum()), "n": len(w)})
HI = pd.DataFrame(HI)
det = T2[T2.cond == "red_contblue"]
for k in (2, 3):
    print(f"PSP detected during blue light at {k} SD: {int(det[f'detected_k{k}'].sum())} of {len(det)} cells; "
          f"with red alone: {int(T2[T2.cond == 'red'][f'detected_k{k}'].sum())} of {len(det)}")
HI["holds"] = (HI.p < 0.05) & (HI.mean_red_during_blue < HI.mean_red)
print(HI.round(4).to_string())
print("\nper cell (amplitude reading a, mV):")
print(T2.pivot(index=["subject", "cell"], columns="cond", values="amp_a").round(2))

**Result: partially reproduced.** A PSP is detected during constant blue light in all 13 cells. The amplitude is
smaller during blue light (mean 4.1 against 8.1 mV, paired t(12) = 3.23, p = 0.0072, smaller in 11 of 13; reading b p = 0.0083, 10 of 13), but the
area is not significantly smaller (p = 0.26 and 0.16, smaller in 12 and 11 of 13). The paper finds both smaller.

## In vivo imaging claims (blocked)

fig7e, fig7gh, fig7j, fig7i, fig7m, fig7-s1no, fig7-s1l and fig7-s2 all need, per session, when the
mouse interacted with the social and the object cup, or its time on each side (Methods: SLEAP pose
with 15 keypoints, the nose within 1.3 cup diameters and the cup in a 90 degree cone). The imaging
files hold the CNMF-E traces and footprints, the raw movie and a link to the behaviour video, and
nothing else: no pose, no interaction or position series, no epochs for the habituation and test
periods, and no record of which cup held the juvenile. The SLEAP model is not published. The data
check above shows what is there, and how the ROI counts compare with the paper's cell counts.

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the
differences from the paper's method.

| Claim | Outcome |
|---|---|
| fig6df-epsp-ipsp-at-rest | reproduced (cell counts differ: 15, 40, 56 against 19, 36, 48) |
| fig6gi-firing-responses | partially reproduced (3, 13, 13 firing cells against 5, 20, 17) |
| fig6j-epsp-amplitude-region | reproduced under the adopted reading (cells with an EPSP, mix included) |
| fig6k-ipsp-amplitude-region | not reproduced |
| fig6-s1de-monosynaptic | blocked: TTX/4AP sweeps not marked |
| fig6no-cea-clusters | partially reproduced |
| fig6-s1h-cea-cluster-properties | reproduced |
| fig6pq-blp-clusters | partially reproduced |
| fig6-s1i-blp-cluster-properties | reproduced |
| fig6-s1jk-cluster-voltage-area | reproduced |
| fig6-s1ln-combined-clusters | not reproduced |
| fig7-s1ef-red-vs-blue | reproduced |
| fig7-s1hi-red-during-blue | partially reproduced |
| In vivo imaging (8 claims) | blocked: no pose, interactions or cup sides |
| Figures 1 to 5 and 8 (17 claims) | blocked: data not in the dataset |

How the analysis differs from the paper's:

- The light time is not in the files; the first pulse is taken 5.468 s into each 30 s light sweep, where the
  responses start. Cells recorded only with other light protocols (20 s and 60 s sweeps, light at other times) are
  left out, and the counts of cells at rest and firing differ from the paper's.
- The paper's EPSP and IPSP were measured by hand in Clampfit with no stated threshold; here a deflection is
  detected beyond 2 or 3 SD of the 5 s baseline of the average trace, the criterion's readings. This finds more
  IPSPs than the paper reports.
- Capacitance is taken from the 5 mV seal-test pulses by the Q-method, not from the paper's 250 ms step and
  MATLAB code, which are not in the files or published.
- The ChrimsonR sweeps are matched to the light conditions by their descriptions.

What was found about the dataset:

- The light timing is not recorded; the stimulus series of the light sweeps are zero, and the protocol is given
  only in free text.
- In current clamp the injected current (ch-1) is labelled volts; the seal test is stored as an IZeroClampSeries but
  is voltage clamp. One cell (P10) has a membrane potential of -18 mV and a ch-1 outside any plausible current.
- No sweep is marked as recorded in TTX/4AP.
- Two cells named P (the BLP naming) are labelled BNST.
- The imaging traces are stated at 25 Hz, but their length matches the behaviour video only at 20 Hz, the Methods'
  rate. The imaging files' virus, stimulus and experiment descriptions describe another experiment (GRIN lens in
  the mPFC, Chrimson in the BLA, 589 nm light at 20 Hz for 5 min).
- The imaging files hold about twice as many ROIs as the paper's cell counts (869, 913 and 854 against 429, 441 and
  484 for the ChrimsonR mice), with nothing marking the ROIs the paper kept, and no pose, interaction or cup-side
  data, so none of the in vivo findings can be tested.